# Wrongway AlphaZero - `standard` run on Kaggle

Clones the repo, trains the `standard` preset (2000 bootstrap games, 20 iterations x 200 self-play
games at 100 MCTS simulations) and mirrors the run to the Hugging Face repo `henrytang785/az_agent`.

**Before running**
- Settings: Accelerator **GPU T4 x2**, Internet **on**.
- Add-ons -> Secrets: `HF_STORAGE_TOKEN` (HF write token) and `ON_KAGGLE`.
- Use **Save Version -> Save & Run All (Commit)** so it keeps running after you close the tab.

**One session is not enough.** Self-play runs on the CPU (Kaggle gives 4 cores), so the run needs
roughly 2-3 sessions of 12 h. The pipeline pauses itself before an iteration that would cross
`STOP_AFTER_HOURS`, pushes everything to Hugging Face, and the next run of this notebook pulls it
back and continues. Just commit the notebook again until the last cell prints the benchmark.

**What the GPUs do.** Only the training step (p02, p03 train, p04 distill) runs on GPU, wrapped in
`torch.nn.DataParallel` across both T4s. The net is small (651->256->128), so most wall time is
CPU self-play and gate games.

## 1. Hugging Face login

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
onkaggle = bool(user_secrets.get_secret("ON_KAGGLE"))
hftoken = user_secrets.get_secret("HF_STORAGE_TOKEN")

if hftoken:
    login(hftoken)
else:
    print("Error: HF_TOKEN (HUGGINGFACE TOKEN) environment variable not set.")

## 2. Settings

In [ ]:
REPO_URL = "https://github.com/edamleanh/wrongway.git"
BRANCH = "main"
RUN = "std1"                        # run name; keep it the same across sessions to resume
PRESET = "standard"                 # only used when the run does not exist yet
HF_REPO = "henrytang785/az_agent"   # private model repo (created if missing)
STOP_AFTER_HOURS = 11.0             # Kaggle kills the session at 12 h

WORKDIR = "/kaggle/working/wrongway"
RUNS_DIR = "/tmp/az_runs"            # big data shards stay out of the saved notebook output (HF keeps them)

## 3. Clone the repo and install dependencies

In [ ]:
import os, subprocess

if os.path.isdir(os.path.join(WORKDIR, ".git")):
    subprocess.run(["git", "-C", WORKDIR, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", WORKDIR, "reset", "--hard", "origin/" + BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, WORKDIR], check=True)
subprocess.run(["git", "-C", WORKDIR, "log", "--oneline", "-1"], check=True)
os.makedirs(RUNS_DIR, exist_ok=True)
os.chdir(WORKDIR)

In [ ]:
# torch + CUDA are preinstalled on Kaggle; only add what the pipeline needs on top.
!pip install -q "huggingface_hub>=0.23" tensorboard pytest

## 4. Hardware check

In [ ]:
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| GPUs", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print("  ", i, torch.cuda.get_device_name(i))
print("CPU cores:", os.cpu_count(), "-> game workers:", max(1, os.cpu_count() - 1))

## 5. Resume from Hugging Face

Downloads `runs/<RUN>/` from the HF repo if an earlier session pushed it. First session: nothing to pull.

In [ ]:
import sys
sys.path.insert(0, WORKDIR)
os.environ["AZ_RUNS_DIR"] = RUNS_DIR
from training.alphazero.hf_sync import pull

resumed = pull(RUN, HF_REPO, RUNS_DIR)
print("resume" if resumed else "new run")

## 6. Train

Runs phases 1-7 in a subprocess and streams the log. Phases already done are skipped; p03 continues
at the next iteration. The run is pushed to Hugging Face after every phase and every iteration
(`AZ_HF_REPO`). Exit code 3 = paused at the time limit: commit this notebook again to continue.

In [ ]:
manifest = os.path.join(RUNS_DIR, RUN, "manifest.json")
cmd = [sys.executable, "-u", "-m", "training.alphazero.pipeline", "--run", RUN,
       "--set", "stop_after_hours=%s" % STOP_AFTER_HOURS, "--set", "data_parallel=true"]
if not os.path.exists(manifest):
    cmd += ["--preset", PRESET]
env = dict(os.environ, AZ_RUNS_DIR=RUNS_DIR, AZ_HF_REPO=HF_REPO, PYTHONUNBUFFERED="1")
print(" ".join(cmd))

proc = subprocess.Popen(cmd, cwd=WORKDIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
with open(os.path.join(RUNS_DIR, RUN + ".log"), "a") as log:
    for line in proc.stdout:
        print(line, end="")
        log.write(line)
exit_code = proc.wait()
if exit_code == 3:
    print("\nPAUSED at the time limit - everything is on Hugging Face. Commit this notebook again to continue.")
elif exit_code:
    print("\nFAILED with exit code", exit_code, "- see the log above")
else:
    print("\nALL PHASES DONE")

## 7. Status and files on Hugging Face

In [ ]:
subprocess.run([sys.executable, "-m", "training.alphazero.pipeline", "--run", RUN, "--status"],
               cwd=WORKDIR, env=env)
from huggingface_hub import list_repo_files
best = [f for f in list_repo_files(HF_REPO) if f.startswith("best/" + RUN + "/")]
print("\n".join(best))
print("https://huggingface.co/" + HF_REPO)

## 8. Final push (safety net)

The pipeline already pushes after every phase/iteration; this catches anything written after the
last automatic push (for example if a push failed on a network error).

In [ ]:
subprocess.run([sys.executable, "-m", "training.alphazero.hf_sync", "push", "--run", RUN, "--repo", HF_REPO],
               cwd=WORKDIR, env=env)

## 9. (Optional, interactive sessions only) TensorBoard

In a committed run this cell renders nothing useful. To watch from your own machine instead:

```bash
hf download henrytang785/az_agent --include "runs/std1/tb/*" --local-dir az_hf
uv run tensorboard --logdir az_hf/runs/std1/tb
```

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUNS_DIR}